In [ ]:
from pathlib import Path
from typing import cast

import json

# pytorch
import torch
from torch import nn
from torch_geometric.data import Data

from dataset import add_stratified_masks, load_facebook_data
from modules import (
    MLPNetwork,
    MLPNetwork3Layer,
    MLPNetwork4Layer,
    MLPNetwork5Layer,
    MLPNetwork4LayerELU,
    MLPNetwork4LayerLReLU,
)

from train import (
    train_mlp_model,
    classification_metrics,
    plot_training_validation_loss,
    plot_training_validation_accuracy
)

# plotting
from matplotlib import pyplot as plt
import seaborn as sns

## Multi-Layer GNN for Facebook Page Classification
### COMP3710 Working Prototype Project in the Era of Artificial Intelligence
#### Derek McKinney (dm-uq)
#### SN: 45506053

### MLP Tuning
<br>
References:

Facebook Page-Page dataset:
    https://github.com/gayanku/facebook_page-page_dataset

PyG FacebookPagePage:
    https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.FacebookPagePage.html

Config and Constants

In [ ]:
# constants
SEED = 808

# data parameters
TRAIN_PCT = 60
VAL_PCT = 20
TEST_PCT = 20

# MLP model parameters
HIDDEN_CHANNELS = 64
HIDDEN_CHANNELS_32 = 32
HIDDEN_CHANNELS_128 = 128
DROPOUT = 0.5
DROPOUT_03 = 0.3
DROPOUT_055 = 0.55
DROPOUT_06 = 0.6
DROPOUT_07 = 0.7

# loss function
CRITERION = nn.CrossEntropyLoss()

# hyper-parameters
LEARNING_RATE = 1e-3
LEARNING_RATE_5E4 = 5e-4
LEARNING_RATE_2E3 = 2e-3
LEARNING_RATE_1E4 = 1e-4
WEIGHT_DECAY = 0.0
WEIGHT_DECAY_5E5 = 5e-5
WEIGHT_DECAY_1E4 = 1e-4
WEIGHT_DECAY_5E4 = 5e-4
WEIGHT_DECAY_1E3 = 1e-3

EPOCHS_4K = 4000
EPOCHS_2K = 2000

# plot styling
sns.set_theme(
    context="notebook",
    style="whitegrid",
)

# output paths
RESULTS_DIR = Path("results")
CHECKPOINTS_DIR = Path("checkpoints")
FIGURES_DIR = Path("images/mlp")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Television show",
    "Government",
    "Company",
    "Politician",
]

In [ ]:
# Device configuration
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Load the Facebook Graph

In [ ]:
# load the FacebookPagePage dataset
dataset = load_facebook_data()
data = dataset[0]  # get first graph object

# create reproducible stratified train/validation/test splits
add_stratified_masks(
    cast(Data, data),
    train_pct=TRAIN_PCT,
    val_pct=VAL_PCT,
    test_pct=TEST_PCT,
    seed=SEED,
)

# load data onto the device
data = data.to(device)

### Depth Tuning

Ensure the Torch seed rng is reset

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate a 2-Layer 'Baseline' MLP network

In [ ]:
mlp2_model = MLPNetwork(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT,
).to(device)

Define the Loss function and Optimiser for the 2-Layer MLP

In [ ]:
# optimiser
mlp2_optimizer = torch.optim.Adam(
    mlp2_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 2-Layer MLP Candidate

In [ ]:
# perform training and validation of the 2-Layer 'Initial MLP'

mlp2_result = train_mlp_model(
    model=mlp2_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp2_optimizer,
    epochs=EPOCHS_4K,
    print_every=100,
)

Restore best result for 2-Layer MLP evaluation

In [ ]:
# check that a best state exists
if mlp2_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp2_model.load_state_dict(
    mlp2_result["best_state"]
)

mlp2_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp2_logits = mlp2_model(
        data.x,
    )

# calculate validation metrics
mlp2_metrics = classification_metrics(
    logits=mlp2_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 2-Layer MLP evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp2_result['best_epoch']}\n"
    f"Validation loss: {mlp2_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp2_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp2_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp2_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp2_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp2_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp2_metrics['precision'][i]:.4f}  "
        f"R={mlp2_metrics['recall'][i]:.4f}  "
        f"F1={mlp2_metrics['f1'][i]:.4f}  "
        f"n={mlp2_metrics['support'][i]}"
    )

Visualise Validation Loss for the 2-Layer MLP

In [ ]:
mlp2_loss_fig, _ = plot_training_validation_loss(
    history=mlp2_result["history"],
    best_epoch=mlp2_result["best_epoch"],
    title="2-Layer MLP Training and Validation Loss",
)

plt.show()

In [ ]:
mlp2_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_2layer_training_validation_loss_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 2-Layer MLP

In [ ]:
mlp2_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp2_result["history"],
    best_epoch=mlp2_result["best_epoch"],
    title="2-Layer MLP Validation Accuracy",
)

plt.show()

In [ ]:
mlp2_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_2layer_validation_accuracy_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 2-Layer MLP Candidate Results

In [ ]:
# save 2-Layer MLP experiment results
mlp2_results = {
    "model": "MLP",
    "experiment": "depth",
    "num_layers": 2,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_4K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp2_result["best_epoch"],
    "best_val_loss": mlp2_result["best_val_loss"],
    "training_seconds": mlp2_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp2_metrics,
    "history": mlp2_result["history"],
}

with (
    RESULTS_DIR / "mlp_depth_2layer.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp2_results,
        file,
        indent=2,
    )

torch.save(
    mlp2_result["best_state"],
    CHECKPOINTS_DIR / "mlp_depth_2layer.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate a 3-Layer 'Baseline' MLP network

In [ ]:
mlp3_model = MLPNetwork3Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT,
).to(device)

Define the Loss function and Optimiser for the 3-Layer MLP

In [ ]:
# optimiser
mlp3_optimizer = torch.optim.Adam(
    mlp3_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 3-Layer MLP Candidate

In [ ]:
# perform training and validation of the 3-Layer 'Initial MLP'

mlp3_result = train_mlp_model(
    model=mlp3_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp3_optimizer,
    epochs=EPOCHS_4K,
    print_every=100,
)

Restore best result for 3-Layer MLP evaluation

In [ ]:
# check that a best state exists
if mlp3_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp3_model.load_state_dict(
    mlp3_result["best_state"]
)

mlp3_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp3_logits = mlp3_model(
        data.x,
    )

# calculate validation metrics
mlp3_metrics = classification_metrics(
    logits=mlp3_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 3-Layer MLP evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp3_result['best_epoch']}\n"
    f"Validation loss: {mlp3_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp3_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp3_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp3_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp3_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp3_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp3_metrics['precision'][i]:.4f}  "
        f"R={mlp3_metrics['recall'][i]:.4f}  "
        f"F1={mlp3_metrics['f1'][i]:.4f}  "
        f"n={mlp3_metrics['support'][i]}"
    )

Visualise Validation Loss for the 3-Layer MLP

In [ ]:
mlp3_loss_fig, _ = plot_training_validation_loss(
    history=mlp3_result["history"],
    best_epoch=mlp3_result["best_epoch"],
    title="3-Layer MLP Training and Validation Loss",
)

plt.show()

In [ ]:
mlp3_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_3layer_training_validation_loss_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 3-Layer MLP

In [ ]:
mlp3_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp3_result["history"],
    best_epoch=mlp3_result["best_epoch"],
    title="3-Layer MLP Validation Accuracy",
)

plt.show()

In [ ]:
mlp3_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_3layer_validation_accuracy_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 3-Layer MLP Candidate Results

In [ ]:
# save 3-Layer MLP experiment results
mlp3_results = {
    "model": "MLP",
    "experiment": "depth",
    "num_layers": 3,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_4K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp3_result["best_epoch"],
    "best_val_loss": mlp3_result["best_val_loss"],
    "training_seconds": mlp3_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp3_metrics,
    "history": mlp3_result["history"],
}

with (
    RESULTS_DIR / "mlp_depth_3layer.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp3_results,
        file,
        indent=2,
    )

torch.save(
    mlp3_result["best_state"],
    CHECKPOINTS_DIR / "mlp_depth_3layer.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate a 4-Layer 'Baseline' MLP network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP Candidate

In [ ]:
# perform training and validation of the 4-Layer 'Initial MLP'

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_4layer_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_4layer_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP Candidate Results

In [ ]:
# save 4-Layer MLP experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "depth",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_depth_4layer.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_depth_4layer.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate a 5-Layer 'Baseline' MLP network

In [ ]:
mlp5_model = MLPNetwork5Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT,
).to(device)

Define the Loss function and Optimiser for the 5-Layer MLP

In [ ]:
# optimiser
mlp5_optimizer = torch.optim.Adam(
    mlp5_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 5-Layer MLP Candidate

In [ ]:
# perform training and validation of the 5-Layer 'Initial MLP'

mlp5_result = train_mlp_model(
    model=mlp5_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp5_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 5-Layer MLP evaluation

In [ ]:
# check that a best state exists
if mlp5_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp5_model.load_state_dict(
    mlp5_result["best_state"]
)

mlp5_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp5_logits = mlp5_model(
        data.x,
    )

# calculate validation metrics
mlp5_metrics = classification_metrics(
    logits=mlp5_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 5-Layer MLP evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp5_result['best_epoch']}\n"
    f"Validation loss: {mlp5_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp5_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp5_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp5_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp5_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp5_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp5_metrics['precision'][i]:.4f}  "
        f"R={mlp5_metrics['recall'][i]:.4f}  "
        f"F1={mlp5_metrics['f1'][i]:.4f}  "
        f"n={mlp5_metrics['support'][i]}"
    )

Visualise Validation Loss for the 5-Layer MLP

In [ ]:
mlp5_loss_fig, _ = plot_training_validation_loss(
    history=mlp5_result["history"],
    best_epoch=mlp5_result["best_epoch"],
    title="5-Layer MLP Training and Validation Loss",
)

plt.show()

In [ ]:
mlp5_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_5layer_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 5-Layer MLP

In [ ]:
mlp5_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp5_result["history"],
    best_epoch=mlp5_result["best_epoch"],
    title="5-Layer MLP Validation Accuracy",
)

plt.show()

In [ ]:
mlp5_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_depth_5layer_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 5-Layer MLP Candidate Results

In [ ]:
# save 5-Layer MLP experiment results
mlp5_results = {
    "model": "MLP",
    "experiment": "depth",
    "num_layers": 5,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp5_result["best_epoch"],
    "best_val_loss": mlp5_result["best_val_loss"],
    "training_seconds": mlp5_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp5_metrics,
    "history": mlp5_result["history"],
}

with (
    RESULTS_DIR / "mlp_depth_5layer.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp5_results,
        file,
        indent=2,
    )

torch.save(
    mlp5_result["best_state"],
    CHECKPOINTS_DIR / "mlp_depth_5layer.pt",
)

TODO Update outcomes cell to MLP outcomes
The 4-layer MLP performed best on every aggregate validation measure.<br>
Every class attains its highest F1 at four layers.<br>
The 4-layer MLP provides the best precision–recall balance for every individual class.

### Dropout Tuning

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (dropout=0.3) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_03,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (dropout=0.3)

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP (dropout=0.3) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (dropout=0.3)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (dropout=0.3) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (dropout=0.3) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (dropout=0.3)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.3) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_030_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (dropout=0.3)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.3) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_030__validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (dropout=0.3) Candidate Results

In [ ]:
# save 4-Layer MLP (dropout=0.3) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "dropout",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_03,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_dropout_030.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_dropout_030.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (dropout=0.55) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (dropout=0.55)

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP (dropout=0.55) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (dropout=0.55)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (dropout=0.55) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (dropout=0.55) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (dropout=0.55)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.55) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_055_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (dropout=0.55)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.55) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_055_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (dropout=0.55) Candidate Results

In [ ]:
# save 4-Layer MLP (dropout=0.55) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "dropout",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_dropout_055.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_dropout_055.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (dropout=0.6) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_06,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (dropout=0.6)

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP (dropout=0.6) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (dropout=0.6)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (dropout=0.6) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (dropout=0.6) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (dropout=0.6)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.6) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_060_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (dropout=0.6)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.6) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_060_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (dropout=0.6) Candidate Results

In [ ]:
# save 4-Layer MLP (dropout=0.6) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "dropout",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_06,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_dropout_060.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_dropout_060.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (dropout=0.7) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_07,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (dropout=0.7)

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP (dropout=0.7) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (dropout=0.7)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (dropout=0.7) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (dropout=0.7) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (dropout=0.7)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.7) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_070_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (dropout=0.7)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.7) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_070_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (dropout=0.7) Candidate Results

In [ ]:
# save 4-Layer MLP (dropout=0.7) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "dropout",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_07,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_dropout_070.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_dropout_070.pt",
)

TODO: Block was gcn-specific and conditional on optimal gcn epoch! - REVIEW
The optimum for dropout=0.7 lands near the training boundary.<br>
Rerun with epochs=4000 to ensure a performance deterioration is observed beyond the current epochs=2000 best.

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (dropout=0.7) network for 4000 epoch run

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_07,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (dropout=0.7) 4000 epoch run

In [ ]:
# optimiser
mlp4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

#### Evaluate 4-Layer MLP (dropout=0.7) Candidate over 4000 epochs

In [ ]:
# perform training and validation of the 4-Layer MLP (dropout=0.7), 4000e

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_optimizer,
    epochs=EPOCHS_4K,
    print_every=100,
)

Restore best result for 4-Layer MLP (dropout=0.7) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (dropout=0.7) evaluation metrics for the 4000e run


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (dropout=0.7) 4000 epoch run

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.7) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_070_training_validation_loss_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (dropout=0.7)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (dropout=0.7) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_dropout_070_validation_accuracy_4000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (dropout=0.7) Candidate Results from the 4000 epoch run

In [ ]:
# save 4-Layer MLP (dropout=0.7) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "dropout",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_4K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_07,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_dropout_070_4000e.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_dropout_070_4000e.pt",
)

TODO Update outcomes cell to MLP outcomes
Using a 4-layer MLP, the dropout=0.50 and dropout=0.55 form the 'best-performing region' for the metrics used.<br>
0.55 was selected as a compromise, because it preserves essentially the same class-balanced classification<br>
performance as 0.50, while moving modestly toward the lower validation loss observed at stronger dropout.

### Weight Decay Tuning

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (weight_decay=1e-4) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (weight_decay=1e-4)

In [ ]:
# optimiser
mlp4_wd1e4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_1E4,
)

#### Evaluate 4-Layer MLP (weight_decay=1e-4) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (weight_decay=1e-4)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_wd1e4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (weight_decay=1e-4) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (weight_decay=1e-4) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (weight_decay=1e-4)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=1e-4) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_wd1e4_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (weight_decay=1e-4)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=1e-4) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_wd1e4_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (weight_decay=1e-4) Candidate Results

In [ ]:
# save 4-Layer MLP (weight_decay=1e-4) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "weight_decay",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_1E4,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_wd1e4.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_wd1e4.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (weight_decay=5e-4) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (weight_decay=5e-4)

In [ ]:
# optimiser
mlp4_wd5e4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E4,
)

#### Evaluate 4-Layer MLP (weight_decay=5e-4) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (weight_decay=5e-4)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_wd5e4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (weight_decay=5e-4) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (weight_decay=5e-4) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (weight_decay=5e-4)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=5e-4) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_wd5e4_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (weight_decay=5e-4)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=5e-4) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_wd5e4_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (weight_decay=5e-4) Candidate Results

In [ ]:
# save 4-Layer MLP (weight_decay=5e-4) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "weight_decay",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E4,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_wd5e4.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_wd5e4.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (weight_decay=1e-3) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (weight_decay=1e-3)

In [ ]:
# optimiser
mlp4_wd1e3_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_1E3,
)

#### Evaluate 4-Layer MLP (weight_decay=1e-3) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (weight_decay=1e-3)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_wd1e3_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (weight_decay=1e-3) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (weight_decay=1e-3) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (weight_decay=1e-3)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=1e-3) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_wd1e3_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (weight_decay=1e-3)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=1e-3) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_wd1e3_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (weight_decay=1e-3) Candidate Results

In [ ]:
# save 4-Layer MLP (weight_decay=1e-3) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "weight_decay",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_1E3,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_wd1e3.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_wd1e3.pt",
)

The best weight decay metrics are shared between 0 and 1e-4<br>
Weight decay of 5e-4 and 1e-3 have optimal values that would typically warrant 4000-epoch runs,<br>
however, all metrics are worse than for weight decay values of 0 and 1e-4, so longer runs will<br>
not be performed.<br>
Instead, a run using weight decay of 5e-5 will be performed to investigate whether it provides a<br>
useful compromise between the 0 and 1e-4 weight decay values.

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (weight_decay=5e-5) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (weight_decay=5e-5)

In [ ]:
# optimiser
mlp4_wd5e5_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (weight_decay=5e-5) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (weight_decay=5e-5)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_wd5e5_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (weight_decay=5e-5) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (weight_decay=5e-5) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (weight_decay=5e-5)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=5e-5) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_wd5e5_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (weight_decay=5e-5)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (weight_decay=5e-5) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_wd5e5_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (weight_decay=5e-5) Candidate Results

In [ ]:
# save 4-Layer MLP (weight_decay=5e-5) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "weight_decay",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_wd5e5.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_wd5e5.pt",
)

TODO Update outcomes cell to MLP outcomes
The use of weight_decay=5e-5 yields the 'best-performing region' across essentially the whole validation picture.

### Hidden Layers

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (hidden_channels=32) network

In [ ]:
mlp4_hc32_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS_32,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (hidden_channels=32)

In [ ]:
# optimiser
mlp4_hc32_optimizer = torch.optim.Adam(
    mlp4_hc32_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (hidden_channels=32) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (hidden_channels=32)

mlp4_result = train_mlp_model(
    model=mlp4_hc32_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_hc32_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (hidden_channels=32) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_hc32_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_hc32_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_hc32_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (hidden_channels=32) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (hidden_channels=32)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (hidden_channels=32) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_hc32_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (hidden_channels=32)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (hidden_channels=32) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_hc32_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (hidden_channels=32) Candidate Results

In [ ]:
# save 4-Layer MLP (hidden_channels=32) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "hidden_channels",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS_32,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_hc32.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_hc32.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (hidden_channels=128) network

In [ ]:
mlp4_hc128_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS_128,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (hidden_channels=128)

In [ ]:
# optimiser
mlp4_hc128_optimizer = torch.optim.Adam(
    mlp4_hc128_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (hidden_channels=128) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (hidden_channels=128)

mlp4_result = train_mlp_model(
    model=mlp4_hc128_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_hc128_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (hidden_channels=128) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_hc128_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_hc128_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_hc128_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (hidden_channels=128) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (hidden_channels=128)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (hidden_channels=128) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_hc128_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (hidden_channels=128)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (hidden_channels=128) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_hc128_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (hidden_channels=128) Candidate Results

In [ ]:
# save 4-Layer MLP (hidden_channels=128) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "hidden_channels",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS_128,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_hc128.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_hc128.pt",
)

TODO Update outcomes cell to MLP outcomes
Reducing hidden width from 64 to 32 substantially slowed convergence and reduced validation performance.<br>
Best validation loss occurred near the 2000-epoch boundary, but classification metrics plateaued below<br>
those of the 64-channel model, so a 4000e run is unlikely to be of value.<br>
128 channels reduced validation cross-entropy and slightly increased AUROC, but without meaningful<br>
improvement in accuracy, macro-F1, or per-class performance, while training cost increased substantially.<br>
Therefore, proceeding with 64 hidden channels.

### Learning Rate

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (learning_rate=5e-4) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (learning_rate=5e-4)

In [ ]:
# optimiser
mlp4_lr5e4_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE_5E4,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (learning_rate=5e-4) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (learning_rate=5e-4)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_lr5e4_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (learning_rate=5e-4) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (learning_rate=5e-4) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (learning_rate=5e-4)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (learning_rate=5e-4) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_lr5e4_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (learning_rate=5e-4)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (learning_rate=5e-4) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_lr5e4_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (learning_rate=5e-4) Candidate Results

In [ ]:
# save 4-Layer MLP (learning_rate=5e-4) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "learning_rate",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE_5E4,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_lr5e4.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_lr5e4.pt",
)

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (learning_rate=2e-3) network

In [ ]:
mlp4_model = MLPNetwork4Layer(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (learning_rate=2e-3)

In [ ]:
# optimiser
mlp4_lr2e3_optimizer = torch.optim.Adam(
    mlp4_model.parameters(),
    lr=LEARNING_RATE_2E3,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (learning_rate=2e-3) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (learning_rate=2e-3)

mlp4_result = train_mlp_model(
    model=mlp4_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_lr2e3_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (learning_rate=2e-3) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (learning_rate=2e-3) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (learning_rate=2e-3)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (learning_rate=2e-3) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_lr2e3_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (learning_rate=2e-3)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (learning_rate=2e-3) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_lr2e3_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (learning_rate=2e-3) Candidate Results

In [ ]:
# save 4-Layer MLP (learning_rate=2e-3) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "learning_rate",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE_2E3,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_lr2e3.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_lr2e3.pt",
)

TODO Update outcomes cell to MLP outcomes
The reduced learning rate of 5e-4 delayed convergence while producing weaker validation metrics.<br>
The increased learning rate of 2e-3 accelerated convergence but did not improve validation performance.<br>
The existing learning rate of 1e-3 provides the best balance of convergence rate and predictive performance.<br>
The 1e-3 learning rate will be retained.

### Activation Function

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (activation: ELU) network

In [ ]:
mlp4_elu_model = MLPNetwork4LayerELU(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (activation: ELU)

In [ ]:
# optimiser
mlp4_elu_optimizer = torch.optim.Adam(
    mlp4_elu_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (activation: ELU) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (activation: ELU)

mlp4_result = train_mlp_model(
    model=mlp4_elu_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_elu_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (activation: ELU) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_elu_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_elu_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_elu_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (activation: ELU) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (activation: ELU)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (activation: ELU) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_elu_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (activation: ELU)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (activation: ELU) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_elu_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (activation: ELU) Candidate Results

In [ ]:
# save 4-Layer MLP (activation: ELU) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "activation",
    "activation": "ELU",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_elu.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_elu.pt",
)

Reset the Torch seed rng

In [ ]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Instantiate the 4-Layer MLP (activation: L-ReLU) network

In [ ]:
mlp4_lrelu_model = MLPNetwork4LayerLReLU(
    in_channels=data.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=dataset.num_classes,
    dropout=DROPOUT_055,
).to(device)

Define the Loss function and Optimiser for the 4-Layer MLP (activation: L-ReLU)

In [ ]:
# optimiser
mlp4_lrelu_optimizer = torch.optim.Adam(
    mlp4_lrelu_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY_5E5,
)

#### Evaluate 4-Layer MLP (activation: L-ReLU) Candidate

In [ ]:
# perform training and validation of the 4-Layer MLP (activation: L-ReLU)

mlp4_result = train_mlp_model(
    model=mlp4_lrelu_model,
    features=data.x,
    labels=data.y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    criterion=CRITERION,
    optimizer=mlp4_lrelu_optimizer,
    epochs=EPOCHS_2K,
    print_every=100,
)

Restore best result for 4-Layer MLP (activation: L-ReLU) evaluation

In [ ]:
# check that a best state exists
if mlp4_result["best_state"] is None:
    raise RuntimeError(
        "No model state was recorded during training."
    )

# restore the checkpoint with the lowest validation loss
mlp4_lrelu_model.load_state_dict(
    mlp4_result["best_state"]
)

mlp4_lrelu_model.eval()

# forward pass at the selected checkpoint
with torch.inference_mode():
    mlp4_logits = mlp4_lrelu_model(
        data.x,
    )

# calculate validation metrics
mlp4_metrics = classification_metrics(
    logits=mlp4_logits,
    labels=data.y,
    mask=data.val_mask,
)

Print 4-Layer MLP (activation: L-ReLU) evaluation metrics


In [ ]:
# print compact result summary
print(
    f"Best epoch:      {mlp4_result['best_epoch']}\n"
    f"Validation loss: {mlp4_result['best_val_loss']:.5f}\n"
    f"Accuracy:        {mlp4_metrics['accuracy']:.4f}\n"
    f"Macro precision: {mlp4_metrics['macro_precision']:.4f}\n"
    f"Macro recall:    {mlp4_metrics['macro_recall']:.4f}\n"
    f"Macro F1:        {mlp4_metrics['macro_f1']:.4f}\n"
    f"Macro AUROC:     {mlp4_metrics['macro_auroc']:.4f}"
)

In [ ]:
# print per-class metrics
print("\nPer-class validation metrics")

for i, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:16} "
        f"P={mlp4_metrics['precision'][i]:.4f}  "
        f"R={mlp4_metrics['recall'][i]:.4f}  "
        f"F1={mlp4_metrics['f1'][i]:.4f}  "
        f"n={mlp4_metrics['support'][i]}"
    )

Visualise Validation Loss for the 4-Layer MLP (activation: L-ReLU)

In [ ]:
mlp4_loss_fig, _ = plot_training_validation_loss(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (activation: L-ReLU) Training and Validation Loss",
)

plt.show()

In [ ]:
mlp4_loss_fig.savefig(
    FIGURES_DIR
    / "mlp_lrelu_training_validation_loss_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Visualise Validation Accuracy for the 4-Layer MLP (activation: L-ReLU)

In [ ]:
mlp4_accuracy_fig, _ = plot_training_validation_accuracy(
    history=mlp4_result["history"],
    best_epoch=mlp4_result["best_epoch"],
    title="4-Layer MLP (activation: L-ReLU) Validation Accuracy",
)

plt.show()

In [ ]:
mlp4_accuracy_fig.savefig(
    FIGURES_DIR
    / "mlp_lrelu_validation_accuracy_2000e.png",
    dpi=200,
    bbox_inches="tight",
)

Save the 4-Layer MLP (activation: L-ReLU) Candidate Results

In [ ]:
# save 4-Layer MLP (activation: L-ReLU) experiment results
mlp4_results = {
    "model": "MLP",
    "experiment": "activation",
    "activation": "LeakyReLU",
    "num_layers": 4,
    "seed": SEED,
    "train_pct": TRAIN_PCT,
    "val_pct": VAL_PCT,
    "test_pct": TEST_PCT,
    "epochs": EPOCHS_2K,
    "hidden_channels": HIDDEN_CHANNELS,
    "dropout": DROPOUT_055,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY_5E5,
    "best_epoch": mlp4_result["best_epoch"],
    "best_val_loss": mlp4_result["best_val_loss"],
    "training_seconds": mlp4_result["elapsed"],
    "device": str(device),
    "validation_metrics": mlp4_metrics,
    "history": mlp4_result["history"],
}

with (
    RESULTS_DIR / "mlp_lrelu.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        mlp4_results,
        file,
        indent=2,
    )

torch.save(
    mlp4_result["best_state"],
    CHECKPOINTS_DIR / "mlp_lrelu.pt",
)

TODO Update outcomes cell to MLP outcomes
The 4-layer MLP performance metrics did not change significantly with the use of either<br>
ELU or Leaky ReLU activation functions.<br>
Therefore, ReLU is retained as the established activation.

TODO Update frozen model cell to MLP outcomes
#### 'Frozen' MLP Model:

4 layers, ReLU, Adam<br>
dropout = 0.55<br>
weight_decay = 5e-5<br>
64 hidden channels<br>
learning_rate = 1e-3<br>

checkpoint = minimum validation cross-entropy